# MIPDS Milestone 4 — Public Exercise Recognition
**Aaron Tobias · CSC 5352 Deep Learning**

**Status: prepared and unexecuted. The model has not yet been trained and no scores have been invented.**

This temporary vision task recognizes **seven exercise types** from sampled side-view video frames. It uses the public dataset's existing exercise labels, so you do not need to record videos or label every frame to begin. This is a deliberate change from the proposed three-class squat-position task. Exercise recognition can help organize trainer review; it does not establish squat position, form quality, joint geometry, or completed repetitions. Later personal recordings will support evaluation of the domain change and adaptation to a more specific squat task.

The notebook downloads one public archive, imports exercise and participant metadata, extracts labeled frames, freezes an ImageNet ResNet-50, trains the new head, selectively fine-tunes the final two residual stages, and audits 20 validation images. The final checkpoint and original split feed the companion Milestone 5 notebook.

**In Colab:** choose a GPU runtime, run cells in order, then replace the journal placeholders with your actual observations. Both stages and the audit must produce rendered outputs before submission.


## Public source and label meaning

Source: Sai Prashanth, Duddela; Kamath, Priya; Bhandary, Rakshith; Pai, S Ashlesh; PN, Pratham; P, Advith; Mashetty, Abhishek (2026). *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*. Mendeley Data, Version 3. [DOI: 10.17632/kgbb3yn47p.3](https://data.mendeley.com/datasets/kgbb3yn47p/3). License: **CC BY 4.0**. Credit the creators and disclose that this notebook selects side views, samples/resizes frames, and maps category names.

The publisher describes 26 participants, seven exercise categories, and front, side, and diagonal views. Its README specifies `subject_<ID>_<exercise>_<quality>_<view>.mp4`, such as `subject_005_push_up_good_front.mp4`. Code checks the actual archive contents; published participant totals are not substituted for observed counts. Download only **Dataset Subject-wise.zip** (about 2.9 GB); the other two archives reorganize the same videos and should not be combined as extra samples.

| Source category | Class used by this notebook | Exercise described by the publisher |
|---|---|---|
| `abs` | `dumbbell_side_bend` | Standing dumbbell side bend |
| `back` | `bent_over_row` | Bent-over dumbbell row |
| `bicep_curl` | `bicep_curl` | Alternating dumbbell bicep curl |
| `push_up` | `push_up` | Standard push-up |
| `shoulder` | `shoulder_press` | Standing dumbbell shoulder press |
| `squat` | `squat` | Bodyweight squat |
| `tricep` | `triceps_extension` | Overhead dumbbell triceps extension |

The source's `good`/`bad` field is retained as metadata and both categories are included. It is **not a prediction target or a clinical/form judgment** in this experiment. Each sampled frame inherits its clip's exercise label, which is weak frame-level supervision: resting or transitional poses can be ambiguous. Trimming the first/last 10% reduces some idle frames but does not prove every remaining frame is distinctive. Review representative frames and revisit this label limitation when analyzing failures.

All clips/views from a participant stay in one split. Actual recording-session dates are unavailable, so one conservative collection-group ID is assigned per participant; it is not claimed to be a known physical recording session. This prevents person leakage, while shared gym backgrounds and the restricted side-view setup still limit generalization.


In [ ]:
import importlib.util
import subprocess
import sys
packages = ["pandas", "pillow", "matplotlib", "scikit-learn>=1.4,<2", "opencv-python-headless"]
if importlib.util.find_spec("tensorflow") is None:
    packages.append("tensorflow>=2.16,<3")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import shutil
import importlib.metadata
import re
import zipfile
import zlib
import urllib.request
from pathlib import PurePosixPath
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from IPython.display import display, clear_output, Markdown
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

SEED = 42
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
VERSIONS = {name: importlib.metadata.version(name) for name in
            ["tensorflow", "numpy", "pandas", "pillow", "scikit-learn", "opencv-python-headless"]}
display(pd.Series(VERSIONS, name="installed_version"))


In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path("/content/mipds_public_exercises")  # Use this same ROOT in Milestone 5.
VIDEO_DIR = ROOT / "videos"
FRAME_DIR = ROOT / "frames"
OUTPUT = ROOT / "milestone4_outputs"
ARCHIVE_PATH = ROOT / "Dataset_Subject_wise.zip"
VIDEO_MANIFEST = ROOT / "video_manifest.csv"
FRAME_LABELS = ROOT / "frame_labels.csv"
CLASSIFICATION_MANIFEST = ROOT / "classification_manifest.csv"
for directory in [ROOT, VIDEO_DIR, FRAME_DIR, OUTPUT]:
    directory.mkdir(parents=True, exist_ok=True)

DATASET_PAGE = "https://data.mendeley.com/datasets/kgbb3yn47p/3"
DATASET_DOI = "10.17632/kgbb3yn47p.3"
ARCHIVE_URL = "https://data.mendeley.com/public-files/datasets/kgbb3yn47p/files/b878bcef-7bbe-4829-9f02-f4e9bb83ed55/file_downloaded"
# Published checksum observed in the Subject-wise archive's file-information panel.
ARCHIVE_SHA256 = "6f6bfda33f094ba13e8509b1cd9a5c1e479062fd0e7749f6b6713db7d588ef84"
DOWNLOAD_PUBLIC_ARCHIVE = True  # False if you already uploaded it to ARCHIVE_PATH.
VIEW_FILTER = "side"
SOURCE_TO_CLASS = {
    "abs": "dumbbell_side_bend", "back": "bent_over_row", "bicep_curl": "bicep_curl",
    "push_up": "push_up", "shoulder": "shoulder_press", "squat": "squat",
    "tricep": "triceps_extension",
}
CLASSES = list(SOURCE_TO_CLASS.values())
EVALUATION_SCOPE = "unseen_people"
SAMPLE_FPS = 1.0
MAX_FRAMES_PER_VIDEO = 12  # Modest first experiment, sampled across each clip.
TRIM_FRACTION = 0.10
MAX_FRAME_EDGE = 640  # Preserve the whole frame/aspect ratio before model resize.
INPUT_SIZE = (224, 224)
RESIZE_METHOD = "bilinear"
BATCH_SIZE = 8
HEAD_LR = 3e-4
HEAD_MAX_EPOCHS = 20
FINE_TUNE_MAX_EPOCHS = 10
FINE_TUNE_LR = HEAD_LR / 10
LOW_CONFIDENCE_FOR_REVIEW = 0.60


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_path(value):
    path = Path(value)
    return (ROOT / path).resolve() if not path.is_absolute() else path.resolve()

def crc32_file(path):
    checksum = 0
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            checksum = zlib.crc32(block, checksum)
    return checksum & 0xffffffff

if not ARCHIVE_PATH.is_file():
    if not DOWNLOAD_PUBLIC_ARCHIVE:
        raise FileNotFoundError(f"Download Dataset Subject-wise.zip from {DATASET_PAGE} and upload it as {ARCHIVE_PATH}.")
    if shutil.disk_usage(ROOT).free < 8 * 1024**3:
        raise OSError("Allow at least 8 GiB of free disk for the archive, selected clips, frames, and checkpoints.")
    temporary = ARCHIVE_PATH.with_suffix(".download")
    print("Downloading one public Subject-wise archive (about 2.9 GB)…")
    try:
        request = urllib.request.Request(ARCHIVE_URL, headers={"User-Agent": "MIPDS-coursework/1.0"})
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as target:
            total = int(response.headers.get("Content-Length", 0))
            received = 0
            next_report = 256 * 1024**2
            while True:
                block = response.read(1024 * 1024)
                if not block: break
                target.write(block)
                received += len(block)
                if received >= next_report:
                    print(f"Downloaded {received / 1024**3:.2f} GiB" + (f" / {total / 1024**3:.2f} GiB" if total else ""))
                    next_report += 256 * 1024**2
        if sha256_file(temporary) != ARCHIVE_SHA256:
            raise ValueError("Downloaded bytes do not match the publisher's archive checksum.")
        temporary.replace(ARCHIVE_PATH)
    except Exception as exc:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Automatic download failed ({type(exc).__name__}). Download only Dataset Subject-wise.zip from {DATASET_PAGE}, upload it to {ARCHIVE_PATH}, then rerun this cell.") from exc
if sha256_file(ARCHIVE_PATH) != ARCHIVE_SHA256:
    raise ValueError("Archive checksum mismatch. Use the verified Version 3 Subject-wise archive.")
print("Verified the archive against the publisher's SHA-256 checksum.")

FILENAME_PATTERN = re.compile(
    r"^subject_(?P<subject>\d+)_(?P<exercise>abs|back|bicep_curl|push_up|shoulder|squat|tricep)_"
    r"(?P<quality>good|bad)_(?P<view>front|side|diagonal)\.mp4$", re.IGNORECASE)

def parse_public_video_name(member_name):
    # Interpret only the publisher-documented basename; never infer identity from frame appearance.
    name = PurePosixPath(member_name).name
    match = FILENAME_PATTERN.fullmatch(name)
    if not match:
        raise ValueError(f"Unrecognized public video name: {name}. Inspect the source README before changing the parser.")
    values = {key: value.lower() for key, value in match.groupdict().items()}
    subject_id = int(values.pop("subject"))
    if subject_id < 1: raise ValueError("Participant IDs must be positive.")
    values["client_id"] = f"public_subject_{subject_id:03d}"
    values["session_id"] = f"{values['client_id']}_collection"
    values["label"] = SOURCE_TO_CLASS[values["exercise"]]
    return values

if VIEW_FILTER not in {"front", "side", "diagonal"}:
    raise ValueError("Choose one documented camera view.")
video_rows = []
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    members = sorted((member for member in archive.infolist()
                      if not member.is_dir() and member.filename.lower().endswith(".mp4")
                      and "__MACOSX" not in PurePosixPath(member.filename).parts), key=lambda m: m.filename)
    if not members: raise ValueError("The archive contains no MP4 videos.")
    parsed = [(member, parse_public_video_name(member.filename)) for member in members]
    selected = [(member, info) for member, info in parsed if info["view"] == VIEW_FILTER]
    if not selected: raise ValueError("No videos for the requested camera view.")
    required_disk = sum(member.file_size for member, _ in selected) + 2 * 1024**3
    if shutil.disk_usage(ROOT).free < required_disk:
        raise OSError("Insufficient disk for the selected clips and derived outputs.")
    destinations = set()
    for member, info in selected:
        # Build our own safe path; do not extract arbitrary ZIP paths or executable files.
        destination = VIDEO_DIR / info["client_id"] / PurePosixPath(member.filename).name
        if destination in destinations:
            raise ValueError("Duplicate public video basename for one participant.")
        destinations.add(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        cached_verified = (destination.is_file() and destination.stat().st_size == member.file_size
                           and crc32_file(destination) == member.CRC)
        if not cached_verified:
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
        video_rows.append({"path": destination.relative_to(ROOT).as_posix(), "archive_member": member.filename,
                           "client_id": info["client_id"], "session_id": info["session_id"],
                           "label": info["label"], "source_exercise": info["exercise"],
                           "source_form_label": info["quality"], "view": info["view"],
                           "source_dataset": DATASET_DOI})
videos = pd.DataFrame(video_rows).sort_values(["client_id", "source_exercise", "source_form_label", "path"]).reset_index(drop=True)
if set(videos.label) != set(CLASSES):
    raise ValueError("The selected clips must cover all seven configured exercise categories.")
if videos.client_id.nunique() < 3:
    raise ValueError("Need at least three actual participants for train/validation/test grouping.")
videos["video_sha256"] = videos.path.map(lambda path: sha256_file(resolve_path(path)))
if videos.video_sha256.duplicated().any():
    raise ValueError("Duplicate video content found; inspect source membership before training.")
videos.to_csv(VIDEO_MANIFEST, index=False)
display(pd.crosstab(videos.source_exercise, videos.source_form_label))
print(f"Selected {len(videos)} {VIEW_FILTER}-view clips from {videos.client_id.nunique()} observed participants.")


## Extract frames with inherited exercise labels

Use at most 12 frames per clip, spread across its central 80%. Preserve the whole frame, downsize to at most 640 pixels on its longest edge, and save PNGs before the model's 224×224 resize. Each selected frame inherits the clip's existing exercise label; no pose model or new annotation is used to manufacture a target. Inspect the preview and the failure audit for idle/ambiguous frames and background shortcuts. A larger frame count is not a larger number of independent participants.

Saved dataset/split assignments are immutable within this experiment. Changing sampling, views, or labels requires an explicitly new project folder/run rather than quietly editing a validation set after seeing its score.


In [ ]:
def sample_frame_indices(fps, frame_count, sample_fps, max_frames):
    if not np.isfinite(fps) or fps <= 0 or frame_count < 1 or sample_fps <= 0 or max_frames < 1:
        raise ValueError("Need valid video FPS, frame count, positive sample rate, and positive frame cap.")
    step = max(1, int(round(fps / sample_fps)))
    indices = np.arange(0, int(frame_count), step, dtype=int)
    if len(indices) > max_frames:
        indices = indices[np.linspace(0, len(indices)-1, max_frames, dtype=int)]
    return indices

if not 0 <= TRIM_FRACTION < 0.5:
    raise ValueError("Trim fraction must leave a positive middle portion.")
preparation = {"dataset_doi": DATASET_DOI, "archive_sha256": ARCHIVE_SHA256,
               "view": VIEW_FILTER, "classes": CLASSES, "source_to_class": SOURCE_TO_CLASS,
               "sample_fps": SAMPLE_FPS, "max_frames_per_video": MAX_FRAMES_PER_VIDEO,
               "trim_fraction": TRIM_FRACTION, "max_frame_edge": MAX_FRAME_EDGE}
preparation_path = ROOT / "dataset_preparation_record.json"
if preparation_path.exists():
    if json.loads(preparation_path.read_text()) != preparation:
        raise ValueError("Dataset preparation settings changed. Use a new ROOT for a new experiment.")
else:
    preparation_path.write_text(json.dumps(preparation, indent=2), encoding="utf-8")
records = []
for video in videos.itertuples(index=False):
    capture = cv2.VideoCapture(str(resolve_path(video.path)))
    if not capture.isOpened(): raise ValueError(f"Could not open {video.path}.")
    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        trim = int(frame_count * TRIM_FRACTION)
        usable_count = frame_count - 2 * trim
        indices = sample_frame_indices(fps, usable_count, SAMPLE_FPS, MAX_FRAMES_PER_VIDEO) + trim
        video_id = video.video_sha256[:16]
        directory = FRAME_DIR / video_id
        directory.mkdir(exist_ok=True)
        for frame_index in indices:
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            okay, bgr = capture.read()
            if not okay: raise ValueError(f"Could not decode frame {frame_index} from {video.path}.")
            height, width = bgr.shape[:2]
            scale = min(1.0, MAX_FRAME_EDGE / max(height, width))
            if scale < 1.0:
                bgr = cv2.resize(bgr, (max(1, round(width*scale)), max(1, round(height*scale))), interpolation=cv2.INTER_AREA)
            frame_id = f"{video_id}_{frame_index:08d}"
            path = directory / f"{frame_id}.png"
            if not path.exists() and not cv2.imwrite(str(path), bgr): raise OSError("Could not save a frame.")
            records.append({"frame_id": frame_id, "path": path.relative_to(ROOT).as_posix(),
                            "video_path": video.path, "client_id": video.client_id, "session_id": video.session_id,
                            "frame_index": int(frame_index), "timestamp_seconds": float(frame_index / fps),
                            "label": video.label, "source_exercise": video.source_exercise,
                            "source_form_label": video.source_form_label, "view": video.view,
                            "source_dataset": DATASET_DOI})
    finally:
        capture.release()
frames = pd.DataFrame(records)
if frames.empty or frames.frame_id.duplicated().any():
    raise ValueError("Need distinct extracted frame IDs.")
frames.to_csv(FRAME_LABELS, index=False)
print(f"Prepared {len(frames)} exercise-labeled frames; no manual per-frame labels are required.")
display(frames.groupby("label").agg(images=("path", "size"), people=("client_id", "nunique")))


## Save a split by participant

Assign whole participants to approximately 70% training, 15% validation, and 15% test using a fixed seed; small group counts cause rounding. All clips and frames from each participant remain together. Require all seven exercise labels in training and validation and at least 20 validation frames. Reuse the saved classification manifest on reruns and leave the test frames unopened during training and these milestone comparisons.


In [ ]:
def assign_group_splits(table, evaluation_scope, seed):
    if evaluation_scope not in {"within_person_sessions", "unseen_people"}:
        raise ValueError("Choose within_person_sessions or unseen_people.")
    key = "session_id" if evaluation_scope == "within_person_sessions" else "client_id"
    groups = sorted(table[key].unique())
    if len(groups) < 2:
        raise ValueError(f"Need at least two distinct {key} groups for training and validation.")
    rng = np.random.default_rng(seed)
    groups = rng.permutation(groups).tolist()
    n_val = max(1, round(0.15 * len(groups)))
    n_test = max(1, round(0.15 * len(groups))) if len(groups) >= 3 else 0
    while n_val + n_test >= len(groups):
        if n_test > 1: n_test -= 1
        elif n_val > 1: n_val -= 1
        else: raise ValueError("Insufficient training groups.")
    assignments = {g: "validation" for g in groups[:n_val]}
    assignments.update({g: "test" for g in groups[n_val:n_val + n_test]})
    assignments.update({g: "train" for g in groups[n_val + n_test:]})
    result = table.copy()
    result["split"] = result[key].map(assignments)
    return result

def validate_training_manifest(table, classes, evaluation_scope):
    if not {"train", "validation"}.issubset(set(table.split)) or not set(table.split).issubset({"train", "validation", "test"}):
        raise ValueError("Invalid split assignments.")
    keys = ["session_id", "video_path", "sha256"]
    if evaluation_scope == "unseen_people": keys.append("client_id")
    elif evaluation_scope != "within_person_sessions": raise ValueError("Unknown evaluation scope.")
    for key in keys:
        if (table.groupby(key).split.nunique() > 1).any():
            raise ValueError(f"Data leakage: {key} crosses splits.")
    for split in ["train", "validation"]:
        selected = table.loc[table.split.eq(split)]
        if set(selected.label) != set(classes):
            raise ValueError(f"{split} needs all target labels. Collect/label more complete clips; preserve groups.")
    if len(table.loc[table.split.eq("validation")]) < 20:
        raise ValueError("Need at least 20 validation images for the milestone audit.")

labeled = pd.read_csv(FRAME_LABELS, dtype=str, keep_default_na=False)
if not set(labeled.label).issubset(set(CLASSES)) or labeled.label.eq("").any():
    raise ValueError("Public frames require the imported exercise labels.")
if labeled.empty: raise ValueError("No labeled examples remain.")
labeled["sha256"] = labeled.path.map(lambda p: sha256_file(resolve_path(p)))
if CLASSIFICATION_MANIFEST.exists():
    prior = pd.read_csv(CLASSIFICATION_MANIFEST, dtype=str, keep_default_na=False)
    if set(prior.frame_id) != set(labeled.frame_id):
        raise ValueError("Labeled frame set changed after a split was saved. Explicitly version the dataset/split before retraining.")
    for column in ["label", "path", "client_id", "session_id", "video_path", "sha256"]:
        left = prior.set_index("frame_id")[column].sort_index()
        right = labeled.set_index("frame_id")[column].sort_index()
        if not left.equals(right):
            raise ValueError(f"Saved split has changed {column}; explicitly version data and retrain.")
    manifest = prior
else:
    manifest = assign_group_splits(labeled, EVALUATION_SCOPE, SEED)
    validate_training_manifest(manifest, CLASSES, EVALUATION_SCOPE)
    manifest.to_csv(CLASSIFICATION_MANIFEST, index=False)
validate_training_manifest(manifest, CLASSES, EVALUATION_SCOPE)
display(pd.crosstab(manifest.label, manifest.split))
display(manifest.groupby("split").agg(images=("path", "size"), clients=("client_id", "nunique"),
                                      recording_sessions=("session_id", "nunique")))
print("Evaluation scope:", EVALUATION_SCOPE, "· test images are excluded from fitting, checkpoint selection, and validation audits.")


In [ ]:
# Inspect training frames only here; keep test frames unopened.
preview = manifest.loc[manifest.split.eq("train")].groupby("label", sort=False).sample(n=2, random_state=SEED)
fig, axes = plt.subplots(4, 4, figsize=(15, 13))
for ax in axes.flat: ax.axis("off")
for ax, row in zip(axes.flat, preview.itertuples(index=False)):
    with Image.open(resolve_path(row.path)) as image: ax.imshow(image.convert("RGB"))
    ax.set_title(f"{row.label}\n{row.client_id} · source form: {row.source_form_label}", fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT / "training_data_preview.png", dpi=140)
plt.show()


## Stage 1 — Frozen ImageNet ResNet-50

This seven-class exercise-recognition task uses public labeled recordings, with the number of independent participants shown above. ResNet-50 is selected for an initial Colab/server prototype, named residual stages for selective adaptation, and later spatial-map inspection. A small participant pool still calls for cautious interpretation, even when many correlated frames are available. EfficientNet-B0 remains a reasonable alternative when model size, memory, or latency takes priority; no measured performance advantage is assumed here.

The validation pipeline stretches RGB PNGs to 224×224 with bilinear resize and applies Keras ResNet preprocessing outside the saved model. Mild rotation/contrast augmentation applies only to training. The head is Global Average Pooling → Dropout(0.3) → Dense(7, softmax); train with cross-entropy, Adam, and early stopping. ImageNet download failures stop the run. All backbone weights, including normalization statistics, must remain unchanged in Stage 1.


In [ ]:
label_to_index = {label: index for index, label in enumerate(CLASSES)}
train_table = manifest.loc[manifest.split.eq("train")].copy()
val_table = manifest.loc[manifest.split.eq("validation")].copy()
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.015, fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomContrast(0.10, seed=SEED + 1),
], name="training_augmentation")

def decode_resize(path, label):
    image = tf.io.decode_png(tf.io.read_file(path), channels=3)
    image = tf.image.resize(tf.cast(image, tf.float32), INPUT_SIZE, method=RESIZE_METHOD)
    return image, label

def preprocess_training(images, labels):
    images = tf.clip_by_value(augmentation(images, training=True), 0.0, 255.0)
    return tf.keras.applications.resnet50.preprocess_input(images), labels

def preprocess_validation(images, labels):
    return tf.keras.applications.resnet50.preprocess_input(images), labels

def make_dataset(table, training):
    paths = [str(resolve_path(path)) for path in table.path]
    labels = table.label.map(label_to_index).to_numpy(dtype=np.int32)
    data = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        data = data.shuffle(len(table), seed=SEED, reshuffle_each_iteration=True)
    data = data.map(decode_resize, num_parallel_calls=tf.data.AUTOTUNE)
    data = data.batch(BATCH_SIZE)
    # Serial augmentation mapping gives its seeded random layers a stable call order.
    data = data.map(preprocess_training if training else preprocess_validation,
                    num_parallel_calls=1)
    return data.prefetch(tf.data.AUTOTUNE)

train_data = make_dataset(train_table, training=True)
val_data = make_dataset(val_table, training=False)
counts = train_table.label.map(label_to_index).value_counts()
CLASS_WEIGHTS = {i: float(min(5.0, len(train_table) / (len(CLASSES) * counts[i])))
                 for i in range(len(CLASSES))}
print("Training-only class weights:", CLASS_WEIGHTS)

# A failed ImageNet download is an error. Do not replace pretrained weights with random ones.
backbone = tf.keras.applications.ResNet50(
    include_top=False, weights="imagenet", input_shape=(*INPUT_SIZE, 3))
backbone.trainable = False
inputs = tf.keras.Input(shape=(*INPUT_SIZE, 3), name="preprocessed_image")
# Keep BatchNormalization in inference mode in BOTH stages.
features = backbone(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(features)
x = tf.keras.layers.Dropout(0.3, name="head_dropout")(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax", name="exercise_class")(x)
model = tf.keras.Model(inputs, outputs, name="mipds_exercise_recognition")
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=HEAD_LR),
              loss=loss, metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
if backbone.trainable_weights:
    raise AssertionError("Stage 1 backbone must have no trainable weights.")
model.summary()

def hash_weights(weights):
    digest = hashlib.sha256()
    for weight in weights:
        value = np.asarray(weight.numpy())
        digest.update(str(value.shape).encode())
        digest.update(str(value.dtype).encode())
        digest.update(value.tobytes(order="C"))
    return digest.hexdigest()

frozen_before_stage1 = hash_weights(backbone.weights)
STAGE1_CHECKPOINT = OUTPUT / "m4_stage1_best.keras"
stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(STAGE1_CHECKPOINT), monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4,
                                    start_from_epoch=9, restore_best_weights=True),
    tf.keras.callbacks.TerminateOnNaN(),
]
if HEAD_MAX_EPOCHS < 10:
    raise ValueError("Allocate at least 10 Stage 1 epochs.")
history1 = model.fit(train_data, validation_data=val_data, epochs=HEAD_MAX_EPOCHS,
                     class_weight=CLASS_WEIGHTS, callbacks=stage1_callbacks)
if not all(np.isfinite(value) for value in history1.history["val_loss"]):
    raise RuntimeError("Stage 1 validation loss was non-finite; resolve training before continuing.")
# Use the best validation-loss checkpoint, including epochs before early-stop monitoring began.
model.load_weights(str(STAGE1_CHECKPOINT))
STAGE1_FROZEN_VERIFIED = hash_weights(backbone.weights) == frozen_before_stage1
if not STAGE1_FROZEN_VERIFIED:
    raise AssertionError("A Stage 1 backbone weight changed.")
stage1_eval = model.evaluate(val_data, return_dict=True, verbose=0)
pd.DataFrame(history1.history).assign(epoch=lambda x: np.arange(1, len(x)+1)).to_csv(
    OUTPUT / "stage1_history.csv", index=False)
print(f"Stage 1 selected-checkpoint validation accuracy: {stage1_eval['accuracy']:.4f}")
print("All backbone weights unchanged:", STAGE1_FROZEN_VERIFIED)


## Stage 2 — Selective fine-tuning

Unfreeze Keras `conv4_*` and `conv5_*` (torchvision ResNet `layer3` and `layer4`) while keeping all earlier stages and BatchNormalization frozen. Keep the backbone in inference mode for normalization and recompile with fresh Adam, a tenfold lower initial learning rate, and cosine decay. Both stages select the checkpoint with lowest validation loss; report that checkpoint's accuracy and last-epoch accuracy separately. Stage 1 allocates at least 10 epochs and Stage 2 at least 5 before their early-stopping monitors begin.


In [ ]:
backbone.trainable = True
for layer in backbone.layers:
    layer.trainable = (layer.name.startswith(("conv4_", "conv5_"))
                       and not isinstance(layer, tf.keras.layers.BatchNormalization))
allowed_trainable_layers = [layer.name for layer in backbone.layers if layer.trainable and layer.weights]
if not allowed_trainable_layers or any(not name.startswith(("conv4_", "conv5_"))
                                     for name in allowed_trainable_layers):
    raise AssertionError("Unexpected fine-tuning layers.")
frozen_stage2_weights = [weight for layer in backbone.layers if not layer.trainable for weight in layer.weights]
frozen_before_stage2 = hash_weights(frozen_stage2_weights)
trainable_before_stage2 = hash_weights(backbone.trainable_weights)
if FINE_TUNE_MAX_EPOCHS < 5 or not np.isclose(FINE_TUNE_LR, HEAD_LR / 10):
    raise ValueError("Stage 2 needs at least 5 planned epochs and a tenfold lower initial LR.")
steps_per_epoch = math.ceil(len(train_table) / BATCH_SIZE)
cosine_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=FINE_TUNE_LR,
    decay_steps=steps_per_epoch * FINE_TUNE_MAX_EPOCHS, alpha=0.01)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_schedule),
              loss=loss, metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
STAGE2_CHECKPOINT = OUTPUT / "m4_stage2_best.keras"
history2 = model.fit(train_data, validation_data=val_data, epochs=FINE_TUNE_MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS, callbacks=[
        tf.keras.callbacks.ModelCheckpoint(str(STAGE2_CHECKPOINT), monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, min_delta=1e-4,
                                        start_from_epoch=4, restore_best_weights=True),
        tf.keras.callbacks.TerminateOnNaN(),
    ])
if not all(np.isfinite(value) for value in history2.history["val_loss"]):
    raise RuntimeError("Stage 2 validation loss was non-finite; resolve training before continuing.")
model.load_weights(str(STAGE2_CHECKPOINT))
STAGE2_FROZEN_VERIFIED = hash_weights(frozen_stage2_weights) == frozen_before_stage2
if not STAGE2_FROZEN_VERIFIED:
    raise AssertionError("An earlier-stage or frozen normalization weight changed.")
STAGE2_TRAINABLE_CHANGED = hash_weights(backbone.trainable_weights) != trainable_before_stage2
if not STAGE2_TRAINABLE_CHANGED:
    raise AssertionError("The unfrozen backbone parameters did not change.")
stage2_eval = model.evaluate(val_data, return_dict=True, verbose=0)
pd.DataFrame(history2.history).assign(epoch=lambda x: np.arange(1, len(x)+1)).to_csv(
    OUTPUT / "stage2_history.csv", index=False)
FINAL_CHECKPOINT = OUTPUT / "m4_finetuned.keras"
model.save(str(FINAL_CHECKPOINT))
performance = pd.DataFrame([
    {"stage": "1 — frozen", "epochs_completed": len(history1.epoch),
     "selected_validation_accuracy": float(stage1_eval["accuracy"]),
     "last_epoch_validation_accuracy": float(history1.history["val_accuracy"][-1]),
     "initial_learning_rate": HEAD_LR},
    {"stage": "2 — fine-tuned", "epochs_completed": len(history2.epoch),
     "selected_validation_accuracy": float(stage2_eval["accuracy"]),
     "last_epoch_validation_accuracy": float(history2.history["val_accuracy"][-1]),
     "initial_learning_rate": FINE_TUNE_LR},
])
performance.to_csv(OUTPUT / "training_comparison.csv", index=False)
display(performance)
print(f"Fine-tuning change: {100*(stage2_eval['accuracy']-stage1_eval['accuracy']):+.2f} percentage points")
print("Frozen Stage 2 weights unchanged:", STAGE2_FROZEN_VERIFIED)
print("Unfrozen backbone weights changed:", STAGE2_TRAINABLE_CHANGED)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for history, name in [(history1, "frozen"), (history2, "fine-tuned")]:
    epochs = np.arange(1, len(history.epoch) + 1)
    axes[0].plot(epochs, history.history["val_accuracy"], label=name)
    axes[1].plot(epochs, history.history["val_loss"], label=name)
axes[0].set(xlabel="Epoch within stage", ylabel="Validation accuracy", ylim=(0, 1))
axes[1].set(xlabel="Epoch within stage", ylabel="Validation loss")
for ax in axes: ax.legend(); ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / "training_curves.png", dpi=160)
plt.show()


## Stage 4 — Twenty-image prediction audit

Use a fixed-seed sample of exactly 20 validation frames with at least one of every exercise class. These frames receive no gradient updates, but validation influenced checkpoint selection, so this is not a separate test-set result. Render the table and images; review true exercise, top-1 predicted exercise, and softmax confidence. Confidence below the predeclared 0.60 threshold or an incorrect prediction is a failure-review candidate, not a calibrated risk estimate.

If the main sample contains neither kind of case, search further validation images for a real case, disclose that additional search, and keep the main table at 20 images. Do not invent a mistake. Check whether an apparently wrong result involves an ambiguous/idle frame inheriting a clip label, then discuss that supervision limitation honestly.


In [ ]:
def select_audit_rows(table, classes, count, seed):
    if len(table) < count or count < len(classes):
        raise ValueError("Insufficient images or audit size.")
    rng = np.random.default_rng(seed)
    chosen = []
    for label in classes:
        candidates = table.loc[table.label.eq(label)].index.to_numpy()
        if not len(candidates): raise ValueError(f"No audit examples for {label}.")
        chosen.append(int(rng.choice(candidates)))
    remaining = table.index.difference(chosen).to_numpy()
    chosen += rng.choice(remaining, size=count-len(chosen), replace=False).tolist()
    return table.loc[chosen].reset_index(drop=True)

AUDIT_SELECTION = OUTPUT / "audit_selection.csv"
if AUDIT_SELECTION.exists():
    audit_ids = pd.read_csv(AUDIT_SELECTION, dtype=str).frame_id.tolist()
    if len(audit_ids) != 20 or len(set(audit_ids)) != 20 or not set(audit_ids).issubset(set(val_table.frame_id)):
        raise ValueError("Saved audit must contain 20 unique validation frame IDs.")
    audit_table = val_table.set_index("frame_id", drop=False).loc[audit_ids].reset_index(drop=True)
else:
    audit_table = select_audit_rows(val_table, CLASSES, 20, SEED + 100)
    audit_table[["frame_id"]].to_csv(AUDIT_SELECTION, index=False)
audit_probabilities = model.predict(make_dataset(audit_table, training=False), verbose=0)
predicted_indices = audit_probabilities.argmax(axis=1)
audit = audit_table[["frame_id", "path", "client_id", "session_id", "timestamp_seconds"]].copy()
audit["true_class"] = audit_table.label.to_numpy()
audit["top1_predicted_class"] = [CLASSES[i] for i in predicted_indices]
audit["confidence"] = audit_probabilities.max(axis=1)
audit["incorrect"] = audit.true_class.ne(audit.top1_predicted_class)
audit["low_confidence"] = audit.confidence.lt(LOW_CONFIDENCE_FOR_REVIEW)
audit.to_csv(OUTPUT / "prediction_audit_20.csv", index=False)
display(audit[["frame_id", "true_class", "top1_predicted_class", "confidence", "incorrect", "low_confidence"]]
        .style.format({"confidence": "{:.3f}"}))

export_table = audit[["frame_id", "true_class", "top1_predicted_class", "confidence"]].copy()
export_table["frame_id"] = export_table.frame_id.map(lambda value: value[-12:])
export_table["confidence"] = export_table.confidence.map(lambda value: f"{value:.3f}")
fig, ax = plt.subplots(figsize=(12, 9))
ax.axis("off")
table_artist = ax.table(cellText=export_table.values, colLabels=export_table.columns,
                        loc="center", cellLoc="left", colWidths=[0.21, 0.28, 0.31, 0.15])
table_artist.auto_set_font_size(False)
table_artist.set_fontsize(10)
table_artist.scale(1, 1.7)
for (row, column), cell in table_artist.get_celld().items():
    if row == 0: cell.set_facecolor("#ddebf7"); cell.set_text_props(weight="bold")
    elif row % 2 == 0: cell.set_facecolor("#f2f5f8")
ax.set_title("MIPDS: 20 held-out validation frames — fine-tuned model", pad=12)
fig.tight_layout()
fig.savefig(OUTPUT / "prediction_audit_20.png", dpi=180, bbox_inches="tight")
plt.show()

for start in range(0, 20, 5):
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for ax, item in zip(axes, audit.iloc[start:start+5].itertuples(index=False)):
        with Image.open(resolve_path(item.path)) as image: ax.imshow(image.convert("RGB"))
        ax.set_title(f"{item.frame_id[-12:]}\nTrue: {item.true_class}\nPred: {item.top1_predicted_class} ({item.confidence:.2f})",
                     fontsize=9, color="crimson" if item.incorrect else "black")
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT / f"audit_images_{start+1:02d}_{start+5:02d}.png", dpi=160)
    plt.show()

failure_candidates = audit.loc[audit.incorrect | audit.low_confidence].sort_values(
    ["incorrect", "confidence"], ascending=[False, True])
if failure_candidates.empty:
    print("No incorrect or predeclared low-confidence case in these 20. Further real review is required for Prompt 3.")
else:
    display(failure_candidates[["frame_id", "true_class", "top1_predicted_class", "confidence"]])
val_probabilities = model.predict(val_data, verbose=0)
val_true = val_table.label.map(label_to_index).to_numpy(dtype=int)
val_pred = val_probabilities.argmax(axis=1)
print("Validation balanced accuracy:", balanced_accuracy_score(val_true, val_pred))
print("Validation macro F1:", f1_score(val_true, val_pred, average="macro", zero_division=0))


## Design journal — complete from the actual run

The rationale is a draft grounded in the selected task and architecture. Replace every bracket in the other responses after training and visual review, using your own reasoning and the course's AI-use policy. Each response below has four sentences. The first two sentences in Prompt 4 describe new capabilities; the remaining sentences address limitations and next steps.


### 1. Backbone selection rationale

I chose ResNet-50 for an initial seven-class exercise-recognition task using public fitness recordings while I prepare personal squat videos. Task complexity and dataset size support starting from ImageNet features, freezing the encoder first, and limiting later adaptation; thousands of frames still come from a relatively small participant pool. A Colab/server deployment can accommodate this prototype, and the named residual stages make fine-tuning and spatial-map inspection easier to document. EfficientNet-B0 would be reasonable if memory or deployment latency became the stronger constraint, and I would compare measured accuracy and compute before choosing a deployment backbone.


### 2. Stage 1 vs. Stage 2 performance

The frozen-backbone checkpoint achieved [Stage 1 accuracy]% validation accuracy on [N] frames from [number] held-out participants. The fine-tuned checkpoint achieved [Stage 2 accuracy]%, a change of [signed difference] percentage points on the same split. This [improvement / decline / negligible change] suggests [your interpretation of how well ImageNet features fit the exercise domain]. Validation guided checkpoint selection, and correlated frames, inherited clip labels, and shared gym backgrounds limit what these scores establish about my future client recordings.


### 3. Failure case analysis

In audit frame [ID and timestamp], the source exercise label was [class] and the model predicted [class] with confidence [score]. [Describe the visible evidence and whether this was an incorrect or unexpectedly low-confidence result.] The model may have relied on [specific pose, equipment, clothing, or background cue], or the frame may be ambiguous because it inherited a clip-level exercise label; visual inspection alone cannot prove the cause. I would address this with [a concrete label review, collection, or evaluation change], then check a new held-out participant or personal recording; [disclose any additional failure search beyond the main 20].


### 4. System status reflection

MIPDS can now [describe the observed exercise-recognition capability on the evaluated public recordings] using an ImageNet-initialized backbone. It can also [describe the demonstrated confidence/audit output that supports trainer review]. Its most important remaining limitation is [a limitation supported by the run, such as ambiguous still frames or lack of evaluation on my client environments]. My next priority is to collect personal side-view squat recordings and evaluate or adapt the model for the more specific task without treating exercise recognition as validated form assessment.


## Export the trained model and prepare Milestone 5

The model, source attribution, preprocessing, label order, hashes, and original participant split travel together. Milestone 5 verifies the record before loading the spatial backbone. Its independent COCO detector uses the same 20 audit images by default; selecting a deliberately varied set of 20 domain images is also allowed without changing the classifier split.

Milestone 5 still requires three distinct scene images: a single centered person, multiple people, and a person with another COCO-recognizable object. Use suitable public frames where that content is actually visible or add other permitted domain images. The first row is only a candidate; verify it and fill the other two rows. Source exercise names do not identify COCO scene content automatically.


In [ ]:
run_record = {
    "milestone": 4, "seed": SEED, "versions": VERSIONS,
    "task": {"name": "exercise_recognition", "supervision": "source clip exercise label inherited by sampled frames"},
    "dataset": {"doi": DATASET_DOI, "page": DATASET_PAGE, "version": 3, "license": "CC BY 4.0",
                "archive": "Dataset Subject-wise.zip", "archive_sha256": ARCHIVE_SHA256,
                "camera_view": VIEW_FILTER, "max_frames_per_video": MAX_FRAMES_PER_VIDEO,
                "sample_fps": SAMPLE_FPS, "trim_fraction": TRIM_FRACTION, "max_frame_edge": MAX_FRAME_EDGE,
                "form_label_role": "metadata only; both source form categories included",
                "session_id_role": "conservative participant collection group; actual session dates unknown"},
    "backbone": "resnet50", "pretrained_weights": "imagenet",
    "nested_backbone": backbone.name, "feature_layer": "conv5_block3_out",
    "input_size": list(INPUT_SIZE), "resize_method": RESIZE_METHOD,
    "preprocessing": "resnet50", "preprocessing_location": "outside_saved_model",
    "classes": CLASSES, "evaluation_scope": EVALUATION_SCOPE,
    "checkpoint_path": FINAL_CHECKPOINT.relative_to(ROOT).as_posix(),
    "checkpoint_sha256": sha256_file(FINAL_CHECKPOINT),
    "classification_manifest_sha256": sha256_file(CLASSIFICATION_MANIFEST),
    "split_sizes": manifest.split.value_counts().to_dict(),
    "unique_clients_per_split": manifest.groupby("split").client_id.nunique().to_dict(),
    "unique_sessions_per_split": manifest.groupby("split").session_id.nunique().to_dict(),
    "stages": performance.to_dict(orient="records"),
    "checkpoint_selection": "lowest validation loss within each stage",
    "fine_tuning": {"unfrozen_prefixes": ["conv4_", "conv5_"],
                    "batch_normalization_frozen": True, "backbone_training_argument": False,
                    "schedule": "cosine decay", "initial_learning_rate": FINE_TUNE_LR},
    "frozen_weight_checks": {"stage1": STAGE1_FROZEN_VERIFIED,
                             "stage2": STAGE2_FROZEN_VERIFIED,
                             "stage2_unfrozen_changed": STAGE2_TRAINABLE_CHANGED},
    "audit": {"count": 20, "selection": "seeded class-covered validation sample",
              "incorrect": int(audit.incorrect.sum()), "low_confidence": int(audit.low_confidence.sum()),
              "low_confidence_threshold": LOW_CONFIDENCE_FOR_REVIEW},
    "claim_boundary": "exercise type from still frames; no squat phase, form assessment, diagnosis, or temporal counting",
}
(OUTPUT / "m4_run_record.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
if not (ROOT / "detection_manifest.csv").exists():
    pd.DataFrame({"image_id": audit.frame_id, "path": audit.path,
                  "scene_note": [f"Side-view exercise frame; inherited clip label {label}. Review scene content."
                                 for label in audit.true_class]}).to_csv(ROOT / "detection_manifest.csv", index=False)
if not (ROOT / "heatmap_manifest.csv").exists():
    pd.DataFrame([
        {"image_id": "single_person", "path": audit.path.iloc[0], "scene_type": "single",
         "scene_note": "Candidate side-view exercise frame. Verify one person is visible."},
        {"image_id": "multiple_people", "path": "", "scene_type": "same_class_multiple", "scene_note": ""},
        {"image_id": "person_and_object", "path": "", "scene_type": "mixed_classes", "scene_note": ""},
    ]).to_csv(ROOT / "heatmap_manifest.csv", index=False)
print("Model and run record saved. Complete the remaining two scene rows for Milestone 5.")
print("Same project ROOT in both notebooks:", ROOT)


## Before submitting

- Run the verified public-data import and inspect its observed classes/participant counts and the training-frame preview.
- Run both ImageNet training stages and check the unchanged frozen weights, training histories, and validation comparison.
- Review/export the 20-image audit, identify a real incorrect or unexpectedly low-confidence case, and complete all four journal responses.
- Cite the public dataset, explain the temporary exercise-recognition task and inherited clip-label limitation, and follow the syllabus's AI-use requirements.
- Restart and run cells in order using the saved archive/manifests; confirm the notebook finishes without errors. Download the `.ipynb` **with all output cells rendered**.
- Retain the dataset frames, checkpoint, run record, and manifests together for Milestone 5. Colab `/content` is temporary; use Drive or save the project outputs before ending the session.

The prepared code has been checked for syntax and data-splitting mechanics. TensorFlow training and end-to-end Colab execution remain unrun in this environment; this file is not yet a completed submission. Automatic downloads can fail in some environments; the download cell gives a manual archive-upload route using the same public source and checksum.

References: [Dataset and attribution](https://data.mendeley.com/datasets/kgbb3yn47p/3), [Keras ResNet models](https://keras.io/api/applications/resnet/resnet_models/), [Keras transfer learning](https://keras.io/guides/transfer_learning/), [EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping), and [cosine decay](https://keras.io/api/optimizers/learning_rate_schedules/cosine_decay/).
